# Parcial para la casa · Lagrange con datos reales: Runge, Chebyshev y un agente que escoge los nodos

**Métodos Numéricos · UNAL Manizales · octubre de 2026**

Vas a interpolar la **población de Colombia** de 1960 a 2020 escogiendo solo unos pocos años como nodos, y vas a medir el error contra **todos** los años. Los 61 datos anuales hacen de "verdad"; los años 2021–2025 los guardamos para ver qué pasa al **extrapolar**.

Ritual de siempre: **predice en tu guía → corre la celda → explica la diferencia**. Solo cambias lo que está marcado con `# <-- CAMBIA ESTO`.

### Fuente de los datos
- **Indicador:** Población total (SP.POP.TOTL), Banco Mundial (World Development Indicators), estimaciones basadas en *World Population Prospects* de la ONU. Página: https://data.worldbank.org/indicator/SP.POP.TOTL
- **Archivo que descarga el cuaderno:** https://raw.githubusercontent.com/datasets/population/main/data/population.csv (paquete *datasets/population*, conversión a CSV del dato del Banco Mundial).
- **Licencia:** el paquete se publica bajo *Open Data Commons Public Domain Dedication and License* (PDDL 1.0); el dato original del Banco Mundial es CC BY 4.0.
- **Fecha de consulta:** 15 de octubre de 2026. Ese día los 66 valores de Colombia (1960–2025) coincidían **exactamente** con la API del Banco Mundial (última actualización: 8 oct 2026).
- Si la URL falla o el Banco Mundial revisa sus estimaciones, el cuaderno usa una **copia congelada** del 15 oct 2026 que viene escrita abajo, para que todos obtengan las mismas cifras.

In [ ]:
# Celda 1 · Cargar los datos (URL pública + copia congelada de respaldo)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/datasets/population/main/data/population.csv"

# Copia congelada (consultada el 15 oct 2026). Habitantes, años 1960, 1961, ..., 2025
COL_CONGELADA = [
    15606209, 16095203, 16599029, 17113823, 17637411, 18169050,
    18702688, 19235618, 19766796, 20291942, 20811782, 21319096,
    21819563, 22325374, 22833469, 23344644, 23858810, 24393494,
    24950337, 25515626, 26104498, 26713267, 27338331, 27988173,
    28642690, 29268470, 29879112, 30499467, 31130524, 31776279,
    32440069, 33098372, 33760571, 34441473, 35127410, 35804662,
    36462745, 37121149, 37792165, 38454863, 39089934, 39709262,
    40324680, 40937429, 41543171, 42128977, 42691084, 43235374,
    43758808, 44271541, 44777319, 45259614, 45715810, 46151584,
    46565429, 46969940, 47437512, 48131078, 49024465, 49907985,
    50629997, 51188173, 51737944, 52321152, 52886363, 53425635]
# Perú 1960-2020 (se usa en la Parte III)
PER_CONGELADA = [
    10174127, 10475123, 10784612, 11102099, 11427978, 11761085,
    12101297, 12448814, 12803993, 13166767, 13501237, 13838067,
    14210147, 14582867, 14956552, 15333957, 15716809, 16109806,
    16516059, 16933139, 17360949, 17798805, 18246503, 18705957,
    19174716, 19648204, 20117122, 20582320, 21054702, 21532589,
    22014771, 22483775, 22946340, 23413783, 23881493, 24348919,
    24811171, 25274147, 25738095, 26186504, 26598331, 26964601,
    27288217, 27580490, 27852190, 28101180, 28324491, 28530699,
    28719041, 28896811, 29086019, 29304086, 29550366, 29817919,
    30115826, 30457600, 30866494, 31324637, 31897584, 32449303,
    32838579]
anos_col = np.arange(1960, 2026)
anos_per = np.arange(1960, 2021)

try:
    df = pd.read_csv(URL)
    web = df[df["Country Code"] == "COL"].set_index("Year")["Value"]
    web = np.array([web.get(a, np.nan) for a in anos_col], dtype=float)
    if np.allclose(web, COL_CONGELADA):
        print("Datos descargados de la URL: coinciden con la copia congelada.")
    else:
        print("OJO: la URL trae valores distintos (el Banco Mundial pudo revisarlos).")
        print("Usamos la copia congelada del 15 oct 2026 para que todos tengan las mismas cifras.")
except Exception as e:
    print("No se pudo descargar la URL (", type(e).__name__, "). Usamos la copia congelada.")

pob = np.array(COL_CONGELADA) / 1e6          # en MILLONES de habitantes
ANOS = anos_col[anos_col <= 2020].astype(float)   # 1960, ..., 2020: la "verdad"
POB  = pob[anos_col <= 2020]
ANOS_FUT = anos_col[anos_col > 2020].astype(float) # 2021, ..., 2025: para extrapolar
POB_FUT  = pob[anos_col > 2020]
POB_PER = np.array(PER_CONGELADA) / 1e6
print("Datos de 1960 a 2020:", len(ANOS), "años.  Población 1960:", POB[0], "M   2020:", POB[-1], "M")

**Celda 2.** Las herramientas. No tienes que modificar nada aquí, pero léelas: son las fórmulas de clase.

- `lagrange(xn, yn, t)`: el polinomio de Lagrange P(t) = Σ yⱼ Lⱼ(t), con Lⱼ(t) = ∏_{m≠j} (t − xₘ)/(xⱼ − xₘ).
- `w(xn, t)`: w(t) = (t − x₀)(t − x₁)…(t − xₙ). La calculamos en la variable reescalada **s = (año − 1990)/30**, que lleva [1960, 2020] a [−1, 1], para poder compararla con el teorema de Chebyshev: ningún polinomio mónico de grado k en [−1, 1] tiene máx|·| menor que 2^(1−k).
- `lebesgue(xn, t)`: λ(t) = Σ |Lⱼ(t)|; su máximo es la constante de Lebesgue Λ.
- `nodos_equi(k)` y `nodos_cheb(k)`: escogen k años de los datos. Los de Chebyshev son **los años más cercanos** a los nodos de Chebyshev de [1960, 2020] (no podemos usar 1987,35: solo tenemos años enteros).

In [ ]:
# Celda 2 · Herramientas
def lagrange(xn, yn, t):
    """Polinomio de Lagrange con nodos xn y valores yn, evaluado en t."""
    t = np.atleast_1d(np.asarray(t, dtype=float))
    P = np.zeros_like(t)
    for j in range(len(xn)):
        Lj = np.ones_like(t)
        for m in range(len(xn)):
            if m != j:
                Lj *= (t - xn[m]) / (xn[j] - xn[m])
        P += yn[j] * Lj
    return P

def s(ano):                      # cambio de variable: [1960, 2020] -> [-1, 1]
    return (np.asarray(ano, dtype=float) - 1990) / 30

def w(xn, t):
    """w(t) = producto de (t - x_i), en la variable s."""
    t = np.atleast_1d(s(t)); W = np.ones_like(t)
    for xi in s(xn):
        W *= (t - xi)
    return W

def lebesgue(xn, t):
    """Funcion de Lebesgue: suma de |L_j(t)|."""
    t = np.atleast_1d(np.asarray(t, dtype=float)); S = np.zeros_like(t)
    for j in range(len(xn)):
        Lj = np.ones_like(t)
        for m in range(len(xn)):
            if m != j:
                Lj *= (t - xn[m]) / (xn[j] - xn[m])
        S += np.abs(Lj)
    return S

def nodos_equi(k):
    """k años equiespaciados entre 1960 y 2020 (redondeados al año)."""
    i = np.round(np.linspace(0, len(ANOS) - 1, k)).astype(int)
    return ANOS[i]

def nodos_cheb(k):
    """Los k años más cercanos a los nodos de Chebyshev de [1960, 2020]."""
    i = np.arange(k)
    ideales = 1990 + 30 * np.cos((2 * i + 1) * np.pi / (2 * k))
    return np.sort(np.array([ANOS[np.argmin(np.abs(ANOS - c))] for c in ideales]))

def evaluar(anos_nodos, x=ANOS, y=POB):
    """Interpola con esos años y mide el error contra TODOS los datos."""
    xn = np.array(sorted(anos_nodos), dtype=float)
    yn = np.array([y[np.where(x == a)[0][0]] for a in xn])
    err = np.abs(lagrange(xn, yn, x) - y)
    return xn, yn, err

T_FINA = np.linspace(1960, 2020, 60001)   # malla fina para max|w| y Lebesgue
print("Listo. Ejemplo: nodos de Chebyshev con k = 5 ->", nodos_cheb(5).astype(int))

**Celda 3.** Mira los datos. Fíjate en 2016–2019: el crecimiento anual pasa de unos 0,4 millones a casi 0,9 millones por año (llegada de población migrante desde Venezuela). La curva es suave "a ojo", pero sus derivadas altas no lo son.

In [ ]:
# Celda 3 · Los datos y el crecimiento anual
crec = np.diff(POB)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ANOS, POB, "k.", label="1960-2020 (verdad)")
ax[0].plot(ANOS_FUT, POB_FUT, "x", color="gray", label="2021-2025 (reservados)")
ax[0].set_xlabel("año"); ax[0].set_ylabel("millones"); ax[0].set_title("Población de Colombia"); ax[0].legend()
ax[1].bar(ANOS[1:], crec, color="tab:blue")
ax[1].set_xlabel("año"); ax[1].set_ylabel("millones por año"); ax[1].set_title("Crecimiento anual")
plt.tight_layout(); plt.show()
for a in [2015, 2016, 2017, 2018, 2019]:
    print(a, "crecimiento: %.3f M" % crec[int(a - 1961)])

### Ítem II.1 · Equiespaciados contra Chebyshev con pocos nodos
**Predice en tu guía antes de correr:** con k = 7 años, ¿qué conjunto da menor error máximo, los equiespaciados (1960, 1970, …, 2020) o los más cercanos a Chebyshev? ¿Cuántas veces menor? ¿En qué año crees que estará el peor error?

In [ ]:
# Celda 4 · Ítem II.1
K = 7   # <-- CAMBIA ESTO (solo si el ítem te lo pide)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for nombre, nodos, color in [("equiespaciados", nodos_equi(K), "tab:red"),
                             ("Chebyshev (años más cercanos)", nodos_cheb(K), "tab:green")]:
    xn, yn, err = evaluar(nodos)
    j = np.argmax(err)
    print(f"{nombre:32s} años = {xn.astype(int)}")
    print(f"{'':32s} error máximo = {err.max():.4f} M  (en {int(ANOS[j])})")
    ax[0].plot(T_FINA, lagrange(xn, yn, T_FINA), color=color, label=nombre)
    ax[1].semilogy(ANOS, err + 1e-6, ".-", color=color, label=nombre)
ax[0].plot(ANOS, POB, "k.", ms=3); ax[0].set_title(f"k = {K} nodos"); ax[0].legend(fontsize=8)
ax[1].set_title("|P(año) - dato|  (millones, escala log)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Ítem II.2 · ¿Qué pasa al subir k?
**Predice en tu guía antes de correr:** llena la columna "mi predicción" de la tabla de la guía para k = 7, 13 y 21. Con más datos, ¿el error de los equiespaciados baja, sube o se queda igual? ¿Y el de Chebyshev?

In [ ]:
# Celda 5 · Ítem II.2
K_LISTA = [7, 13, 21]   # <-- CAMBIA ESTO si quieres probar otros k
print(" k | error máx equiespaciados (año) | error máx Chebyshev (año)")
for k in K_LISTA:
    _, _, e_eq = evaluar(nodos_equi(k))
    _, _, e_ch = evaluar(nodos_cheb(k))
    print(f"{k:2d} | {e_eq.max():12.4f} M ({int(ANOS[e_eq.argmax()])})      | "
          f"{e_ch.max():8.4f} M ({int(ANOS[e_ch.argmax()])})")

xn, yn, _ = evaluar(nodos_equi(21))
print("\nCon 21 años equiespaciados, P(1961) =", np.round(lagrange(xn, yn, [1961.0])[0], 3),
      "M   (dato real:", POB[1], "M)")

### Ítem II.3 · ¿Dónde se concentra el error y por qué?
Recuerda la fórmula del error: f(x) − P(x) = f⁽ⁿ⁺¹⁾(ξ)/(n+1)! · w(x). Los nodos solo controlan el factor w.

**Predice en tu guía antes de correr:** con k = 13 años equiespaciados, ¿el error será mayor en los bordes (1960–1965 y 2015–2020) o en el centro (1985–1995)? ¿Cuántas veces más grande crees que es máx|w| en los bordes que en el centro? ¿Y con Chebyshev?

In [ ]:
# Celda 6 · Ítem II.3
K = 13   # <-- CAMBIA ESTO (solo si el ítem te lo pide)
borde_f  = (T_FINA <= 1965) | (T_FINA >= 2015);  centro_f = (T_FINA >= 1985) & (T_FINA <= 1995)
borde_d  = (ANOS <= 1965)   | (ANOS >= 2015);    centro_d = (ANOS >= 1985)   & (ANOS <= 1995)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for nombre, nodos, color in [("equiespaciados", nodos_equi(K), "tab:red"),
                             ("Chebyshev", nodos_cheb(K), "tab:green")]:
    xn, yn, err = evaluar(nodos)
    W = np.abs(w(xn, T_FINA))
    Lam = lebesgue(xn, T_FINA).max()
    print(f"--- {nombre} (k = {K}) ---")
    print(f"error máx en bordes = {err[borde_d].max():.4f} M   en centro = {err[centro_d].max():.4f} M")
    print(f"máx|w| en bordes = {W[borde_f].max():.3e}   en centro = {W[centro_f].max():.3e}"
          f"   razón bordes/centro = {W[borde_f].max() / W[centro_f].max():.1f}")
    print(f"máx|w| en todo [1960, 2020] = {W.max():.3e}    constante de Lebesgue = {Lam:.2f}")
    ax[0].semilogy(T_FINA, W, color=color, label=nombre)
    ax[1].semilogy(ANOS, err + 1e-6, ".-", color=color, label=nombre)
print(f"\nTeorema de Chebyshev: ningún w mónico de grado {K} baja de 2^(1-{K}) = {2.0**(1-K):.3e} en [-1, 1]")
ax[0].axhline(2.0**(1-K), color="gray", ls=":", label="2^(1-k)")
ax[0].set_title("|w(s)|  (s = (año-1990)/30)"); ax[0].legend(fontsize=8)
ax[1].set_title("error por año (millones)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Ítem II.4 · Sé el agente
Ahora tú escoges los 13 años. La meta: **ganarle a Chebyshev**, es decir, error máximo en 1960–2020 menor que el de los 13 años más cercanos a Chebyshev.

**Predice en tu guía antes de correr:** escribe tus 13 años y por qué los escogiste. ¿Cuál año de la lista de Chebyshev moverías primero y hacia dónde?

Reglas: exactamente 13 años distintos, todos entre 1960 y 2020. Puedes correr la celda cuantas veces quieras; anota en la guía cada intento.

In [ ]:
# Celda 7 · Ítem II.4: sé el agente
MIS_ANOS = [1960, 1962, 1965, 1970, 1976, 1983, 1990, 1997, 2004, 2010, 2015, 2018, 2020]   # <-- CAMBIA ESTO

assert len(set(MIS_ANOS)) == 13, "Necesitas exactamente 13 años distintos"
assert all(1960 <= a <= 2020 for a in MIS_ANOS), "Los años deben estar entre 1960 y 2020"

_, _, e_ch = evaluar(nodos_cheb(13))
xn, yn, err = evaluar(MIS_ANOS)
print("Tus años:", sorted(MIS_ANOS))
print(f"Tu error máximo:      {err.max():.4f} M  (en {int(ANOS[err.argmax()])})")
print(f"Error de Chebyshev:   {e_ch.max():.4f} M  (en {int(ANOS[e_ch.argmax()])})")
print(f"Tu máx|w| = {np.abs(w(xn, T_FINA)).max():.3e}    tu constante de Lebesgue = {lebesgue(xn, T_FINA).max():.2f}")
print("¡Le ganaste a Chebyshev!" if err.max() < e_ch.max() else "Todavía no le ganas a Chebyshev. Intenta otra vez.")
plt.figure(figsize=(6, 3.2)); plt.semilogy(ANOS, err + 1e-6, ".-"); plt.title("tu error por año (M)"); plt.show()

## Parte III · Verificar lo que reporta el agente

El agente de la Parte III de la guía es un **escenario inventado para el parcial**, pero sus números no lo son: esta celda los recalcula. Úsala para decidir qué afirmaciones son verdaderas.

In [ ]:
# Celda 8 · Parte III: verificar el reporte del agente
AGENTE = [1960, 1962, 1966, 1970, 1978, 1983, 1989, 1991, 1999, 2005, 2011, 2015, 2020]
VERSION2 = list(range(1960, 1973))          # recompensa = -error en los 13 años usados
VERSION3 = [1968, 1969, 1971, 1975, 1977, 1983, 1992, 1996, 1998, 2001, 2007, 2010, 2016]  # recompensa medida solo en 1970-2010
CHEB = list(nodos_cheb(13).astype(int))
EQUI = list(nodos_equi(13).astype(int))
dentro = (ANOS >= 1970) & (ANOS <= 2010)

print("conjunto   | error máx 1960-2020 (año) | error máx 1970-2010 | máx|w|    | Lebesgue | P(2025)")
for nombre, A in [("equi", EQUI), ("Chebyshev", CHEB), ("agente", AGENTE), ("versión 2", VERSION2), ("versión 3", VERSION3)]:
    xn, yn, err = evaluar(A)
    print(f"{nombre:10s} | {err.max():10.4g} M ({int(ANOS[err.argmax()])})   | {err[dentro].max():10.4g} M        |"
          f" {np.abs(w(xn, T_FINA)).max():.3e} | {lebesgue(xn, T_FINA).max():8.4g} | {lagrange(xn, yn, [2025.0])[0]:10.4g} M")
print("(Con la versión 2 los números gigantes solo valen por su orden de magnitud: el cálculo está muy mal condicionado.)")
print(f"Dato real 2025: {POB_FUT[-1]:.2f} M      2^(1-13) = {2.0**-12:.3e}")

xn, yn, _ = evaluar(VERSION2)
print("\nVersión 2: error en sus propios nodos =", np.abs(lagrange(xn, yn, xn) - yn).max())

print("\nAños del agente por zona: 1960-1970:", sum(a <= 1970 for a in AGENTE),
      "  1985-1995:", sum(1985 <= a <= 1995 for a in AGENTE), "  2010-2020:", sum(a >= 2010 for a in AGENTE))

# Los mismos años aplicados a Perú
for nombre, A in [("agente", AGENTE), ("Chebyshev", CHEB)]:
    _, _, e = evaluar(A, x=ANOS, y=POB_PER)
    print(f"Perú con los años de {nombre:9s}: error máx = {e.max():.4f} M")

xn, yn, _ = evaluar(CHEB)
print(f"\nChebyshev: |w(2025)| = {np.abs(w(xn, [2025.0]))[0]:.3e}  vs  máx|w| en [1960, 2020] = {np.abs(w(xn, T_FINA)).max():.3e}")

**Celda 9 (opcional).** Un "agente" muy simple: empieza en los años de Chebyshev y, mientras pueda, mueve **un** año una posición (±1) si eso baja el error máximo. No es aprendizaje por refuerzo (es búsqueda local), pero muestra que la recompensa −(error máximo) sí se puede mejorar.

In [ ]:
# Celda 9 · Búsqueda local desde Chebyshev (opcional)
actual = list(nodos_cheb(13))
mejor = evaluar(actual)[2].max()
paso = 0
while True:
    candidato = None
    for p in range(len(actual)):
        for d in (-1, 1):
            nuevo = actual[p] + d
            if nuevo < 1960 or nuevo > 2020 or nuevo in actual:
                continue
            prueba = sorted(actual[:p] + [nuevo] + actual[p+1:])
            e = evaluar(prueba)[2].max()
            if e < mejor and (candidato is None or e < candidato[0]):
                candidato = (e, prueba)
    if candidato is None:
        break
    mejor, actual = candidato; paso += 1
print(f"Después de {paso} movimientos: error máx = {mejor:.4f} M con años {[int(a) for a in actual]}")